# 03 — Real data: NGC 5102 MUSE galactic center

NGC 5102 is a lenticular galaxy at 3.66 Mpc observed with VLT/MUSE (0.2 arcsec/spaxel).
This notebook fits the central Voronoi-binned spectrum to recover the line-of-sight
velocity distribution (LOSVD) from the Ca II triplet region.

The spectrum (`bin0105sp.spec`) is the central spatial bin extracted with the
kinextract pipeline and is bundled in `examples/data/muse/`.

**Key aspects of real data:**
- Raw flux (counts, not normalized) — `fit_als_continuum=True` handles this
- Galaxy redshift: `zgal = 0.001556`
- MUSE's per-pixel flux errors are not very reliable, so we override them with a
  uniform S/N-based estimate (`gal_errors=flux/50`) rather than using
  `use_spectrum_errors=True`
- Stellar templates: MUSE Library of Stellar Spectra

In [ ]:
from __future__ import annotations
import tempfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid

from kinextract import FitConfig, run_spectral_fit
from kinextract.fitting import fit_losvd_gauss_hermite

plt.style.use('kinextract.mplstyle')

# Path to bundled example data (adjust if running from a different location)
DATA_DIR = Path('..') / 'data'
MUSE_DIR = DATA_DIR / 'muse'

# ── To use a different spectrum, change this line ────────────────────────────
SPEC_FILE = MUSE_DIR / 'bin0105sp.spec'

## 1. Load and inspect the spectrum

In [2]:
# ── Wavelength grid ───────────────────────────────────────────────────────────
# The .spec file contains integer pixel indices (1–3681); kinextract reconstructs
# wavelengths as: wave = wavemin_full + (pix - 1) * step
WAVEMIN_FULL = 4750.0    # Å, pixel 1
STEP         = 1.25      # Å / pixel
N_PIX        = 3681
wavelength   = WAVEMIN_FULL + np.arange(N_PIX) * STEP   # observed-frame grid

data = np.loadtxt(SPEC_FILE)
pix, flux = data[:, 0].astype(int), data[:, 1]

# MUSE's flux errors are not very reliable, so we can use a simple S/N estimate to set the errors instead
snr = 50.0
ferr = flux / snr

print(f"Spectrum pixels: {len(flux)}")
print(f"Flux range:  {flux.min():.0f} – {flux.max():.0f}  counts")
print(f"Error range: {ferr.min():.0f} – {ferr.max():.0f}  counts")
print(f"Median S/N per pixel: {np.median(flux / ferr):.1f}")

Spectrum pixels: 3681
Flux range:  25340 – 80715  counts
Error range: 507 – 1614  counts
Median S/N per pixel: 50.0


## 2. Quick-look: raw spectrum over the Ca II fitting region

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 7))
ax[0].plot(wavelength, flux, lw=0.8, color='black', alpha=0.9, label='MUSE spectrum')
ax[0].fill_between(wavelength, flux - ferr, flux + ferr, alpha=0.20, color='steelblue', label=r'$1\sigma$ error')
ax[0].set_xlabel(r'$\lambda_{\mathrm{obs}}$ ($\mathrm{\AA}$)')
ax[0].set_ylabel(r'$f_\lambda$ $(\mathrm{10^{-20} erg/s/cm^2/\AA})$')
ax[0].set_title('NGC 5102 — MUSE galactic center')
ax[1].set_xlabel(r'$\lambda_{\mathrm{obs}}$ ($\mathrm{\AA}$)')
ax[1].set_ylabel(r'$f_\lambda$ $(\mathrm{10^{-20} erg/s/cm^2/\AA})$)')


ax[1].plot(wavelength, flux, lw=0.8, color='black', alpha=0.9, label='MUSE spectrum')
ax[1].fill_between(wavelength, flux - ferr, flux + ferr, alpha=0.20, color='steelblue', label=r'$1\sigma$ error')
ax[1].set_xlim(8400, 8750)
ax[1].set_ylim(0.95 * flux[(wavelength > 8400)&(wavelength < 8800)].min(), 
               1.05 * flux[(wavelength > 8400)&(wavelength < 8800)].max())
ax[0].legend(fontsize=9, framealpha=1.0, facecolor='white').set_zorder(20)
ax[1].legend(fontsize=9, framealpha=1.0, facecolor='white').set_zorder(20)
plt.tight_layout()
plt.show()

## 3. FitConfig and run

In [4]:
tmpdir = Path(tempfile.mkdtemp(prefix='kinextract_muse_'))

cfg = FitConfig(
    template_list_file    = str(MUSE_DIR / 'Tlist'),
    template_dir          = str(MUSE_DIR),
    outdir                = str(tmpdir),
    wavemin_full          = WAVEMIN_FULL,  # Å, minimum wavelength of full spectrum (pixel 1)
    step                  = STEP,          # Å / pixel
    wavefitmin            = 8400.0,        # Å  (rest-frame; kinextract applies zgal)
    wavefitmax            = 8750.0,        # Å
    zgal                  = 0.001556,      # redshift estimate from NED (heliocentric)
    losvd_vmin            = -300.0,        # km/s
    losvd_vmax            = +300.0,        # km/s
    fit_als_continuum     = True,
    use_spectrum_errors   = False,         # gal_errors= below takes priority; this is ignored
    xlam_auto             = True,
    xlam_criterion        = 'roughness',   # Allows us to control how smooth the LOSVD is; 'roughness' is a good default for real data
    xlam_smooth_threshold = 0.25,          # Controls how smooth the LOSVD is; smaller = smoother. Default is 0.25.
    sigl                  = 100.0,
    clean                 = True,
    map_maxiter           = 20000,
    print_every           = 10000,
)

# Pass our SNR-based error estimate directly — overrides whatever is in the file
fit = run_spectral_fit(cfg, gal_file=str(SPEC_FILE), gal_errors=ferr)
st  = fit['state']
out = fit['outputs']
b   = out['b']
gp  = out['gp']
print(f"χ²_red = {out['chi2_red']:.3f}")
print(f"xlam used = {st.xlam}")

[     1.06s] ==== spectral fitting START | ../data/muse/bin0105sp.spec ====
[     1.06s] wavefit=[8400.0, 8750.0] z=0.001556 sigl=100.0 xlam=300.0
[     1.06s] fit_als_continuum=True prenorm=False
[     1.06s] START build FitState
[     1.06s] START read spectrum
[     1.06s] gal_errors override applied (raw .spec path)
[     1.06s] fit pixels=280 step=1.25
[     1.06s] END   read spectrum (0.00s)
[     1.06s] START apply masks
[     1.08s] Segment emission mask [8674.1–8748.9 Å]: 2 upward-outlier pixels (>3σ above rolling median)
[     1.08s] Segment emission pre-mask: 2 additional pixels
[     1.08s] END   apply masks (0.01s)
[     1.08s] START read + interpolate templates
[     1.12s] Template fractional error (pooled median): 0.0074
[     1.12s] END   read + interpolate templates (0.05s)
[     1.13s] LOSVD velocity grid from galaxy.params/config: [-300.000, 300.000] km/s, nl=29
[     1.13s] nlosvd reference wavelength: 7037.7992
[     1.13s] START precompute LOSVD + ip map
[     1.

## 4. Gauss-Hermite moments

In [5]:
gh = fit_losvd_gauss_hermite(st.xl, b, fit_h3h4=True)
print(f"V      = {gh['vherm']:+.1f} km/s")
print(f"sigma  = {gh['sherm']:.1f} km/s")
print(f"h3     = {gh['h3']:+.4f}")
print(f"h4     = {gh['h4']:+.4f}")

V      = +22.7 km/s
sigma  = 57.9 km/s
h3     = -0.0243
h4     = -0.0308


## 5. Fit Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('NGC 5102 — MUSE central bin', fontsize=13)

# ── Panel 1: spectral fit ─────────────────────────────────────────────────────
ax = axes[0]
ax.plot(st.x, st.g,  lw=1.0, color='black', label='galaxy', alpha=0.8)
ax.fill_between(st.x, st.g - st.gerr, st.g + st.gerr, alpha=0.20, color='gray', label=r'$1\sigma$ error')
ax.plot(st.x, gp, lw=1.8, color='tomato', label='model', zorder=3)
resid = (st.g - gp) / np.nanmedian(st.gerr[st.gerr < 1e9])
ax.set_ylim(0.9*st.g.min(), 1.1*st.g.max())
ax2 = ax.twinx()
ax2.plot(st.x, resid, lw=0.6, color='grey', alpha=0.6)
ax2.axhline(0, lw=0.5, color='k')
ax2.set_ylabel(r'residual ($\sigma$)', fontsize=8)
ax.set_xlabel('rest-frame wavelength (Å)')
ax.set_ylabel('normalised flux')
ax.set_title(r'Spectral fit  $\chi^2_{\rm red}$ = ' + f"{out['chi2_red']:.2f}")
ax.legend(fontsize=8, framealpha=1.0, facecolor='white').set_zorder(20)

# ── Panel 2: LOSVD ────────────────────────────────────────────────────────────
ax = axes[1]
b_norm = b / trapezoid(b, st.xl)
ax.plot(st.xl, b_norm, lw=2.0, color='black', label='recovered LOSVD')
ax.plot(st.xl, gh['model']/trapezoid(gh['model'], st.xl), lw=1.5, color='tomato', ls='--', label='Gauss-Hermite fit')
ax.axvline(gh['vherm'], lw=1.0, color='tomato', ls=':')
ax.axvline(0,           lw=0.8, color='grey',   ls='--', alpha=0.6)
ax.text(0.77, 0.79,
        f"V   = {gh['vherm']:+.1f} km/s\n"
        fr"$\sigma$" + f"   = {gh['sherm']:.1f} km/s\n"
        f"h3 = {gh['h3']:+.4f}\n"
        f"h4 = {gh['h4']:+.4f}",
        transform=ax.transAxes, fontsize=9, ha='left', va='bottom',
        bbox=dict(boxstyle='round', facecolor='white', alpha=1.0, edgecolor='0.8'))
ax.set_xlabel('velocity (km/s)')
ax.set_ylabel('LOSVD  (km/s)⁻¹')
ax.set_title('Recovered LOSVD — NGC 5102 centre')
ax.legend(fontsize=9, loc='upper left', framealpha=1.0, facecolor='white').set_zorder(20)
ax.set_xlim(cfg.losvd_vmin, cfg.losvd_vmax)

plt.tight_layout()
plt.show()

## 6. Error estimation

Uncertainty on the LOSVD is estimated two ways:

- **Laplace covariance**: Gaussian approximation at the MAP solution — fast, analytical, reliable for Gaussian LOSVDs.
- **Residual bootstrap**: Resamples fit residuals to generate synthetic spectra and re-fits each → non-parametric; preferred for publication uncertainties.

In [7]:
from kinextract import LOSVDErrorEstimator

N_BOOT = 50   # increase to ≥200 for publication

est     = LOSVDErrorEstimator(fit, cfg)
laplace = est.laplace_covariance()
boot    = est.residual_bootstrap(n_bootstrap=N_BOOT, n_jobs=1)
summary = est.summarize(laplace_result=laplace, bootstrap_result=boot)

gh_map = summary['gh_map']
gh_err = summary.get('gh_err_recommended', {})
print("Kinematic moments (bootstrap uncertainties):")
for label, mkey, ekey in [('V', 'vherm', 'gh_vherm'), ('σ', 'sherm', 'gh_sherm'),
                           ('h3', 'h3', 'gh_h3'), ('h4', 'h4', 'gh_h4')]:
    val = gh_map.get(mkey)
    err = gh_err.get(ekey)
    if val is not None:
        err_str = f' ± {err:.2f}' if err is not None else ''
        unit = ' km/s' if label in ('V', 'σ') else ''
        print(f'  {label:5s} = {val:+.2f}{unit}{err_str}')

[LOSVDErrors] Computing Hessian...
[LOSVDErrors] Using JAX gradient-backed Hessian FD (CPU)
[LOSVDErrors] Laplace covariance done in 0.4s. Hessian PD (free params): False. Pinned: 9/29 LOSVD bins, 25/35 template weights. Max projected |grad|: 0.348
[LOSVDErrors] Starting residual bootstrap (n=50, block=1, jobs=1)...
[    10.58s] Ca II mask shift: +0.327 Å  (als_mask_center_shift_A → 0.664 Å)
[    10.58s] ALS absorption-clean iter 1: sigma=1.168, rejected=0, base_pixels=238
[    10.58s] ALS init: lam=1.000e+07 p=0.05 median=3.82e+04 base_pixels=238 line_mask_pixels=38
[    10.58s] ALS outer iteration 1/4
[    10.58s]   protecting Ca II [8487.3, 8510.0] npix=18
[    10.58s]   protecting Ca II [8531.4, 8554.2] npix=18
[    10.58s]   protecting Ca II [8651.2, 8674.4] npix=19
[    10.58s] Cleaning protection: 55 pixels
[    10.58s] START MAP clean iter 1


/var/folders/zp/_mj2qgk5107f3gf_2gnw4gnm0000gs/T/ipykernel_44647/474528249.py:6: RuntimeWarning: Laplace covariance: the largest projected gradient component at the MAP solution is 0.348, above grad_warn_threshold=0.05. This suggests the MAP optimization did not fully converge, which can make the Hessian indefinite and the resulting error bars unreliable (some may be silently near-zero). Consider tightening map_ftol/map_gtol or enabling use_jax_objective before re-fitting.
  laplace = est.laplace_covariance()


[    12.04s] END   MAP clean iter 1 (1.46s)
[    12.04s] Clean converged after 1 iter.
[    12.04s] ALS absorption-clean iter 1: sigma=0.3686, rejected=0, base_pixels=238
[    12.04s]   ALS update: lam=1.000e+07 p=0.05 delta=0.007753 base_pixels=238 line_mask_pixels=38
[    12.04s]   ALS continuum median fractional change = 0.007753
[    12.04s] ALS outer iteration 2/4
[    12.04s] START MAP optimize ALS outer 2
[    12.13s] END   MAP optimize ALS outer 2 (0.09s)
[    12.13s] ALS absorption-clean iter 1: sigma=0.3642, rejected=0, base_pixels=238
[    12.13s]   ALS update: lam=1.000e+07 p=0.05 delta=0.0076 base_pixels=238 line_mask_pixels=38
[    12.13s]   ALS continuum median fractional change = 0.0076
[    12.13s] ALS outer iteration 3/4
[    12.13s] START MAP optimize ALS outer 3
[    12.48s] END   MAP optimize ALS outer 3 (0.35s)
[    12.49s] ALS absorption-clean iter 1: sigma=0.3662, rejected=0, base_pixels=238
[    12.49s]   ALS update: lam=1.000e+07 p=0.05 delta=0.007509 base_pix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('NGC 5102 — MUSE galactic center error estimation', fontsize=13)

b_trap = trapezoid(b, st.xl)
b_map  = b / b_trap

bs       = np.array(boot['b_samples'])
bs_traps = np.array([trapezoid(s, st.xl) for s in bs])[:, np.newaxis]
bs       = bs / bs_traps
b_median = np.median(bs, axis=0)
b_lo     = np.maximum(np.percentile(bs, 16, axis=0), 0.0)
b_hi     = np.percentile(bs, 84, axis=0)
gh_med   = fit_losvd_gauss_hermite(st.xl, b_median, fit_h3h4=True)

# Panel 1: Laplace
ax = axes[0]
b_sig  = laplace['b_err'] / b_trap
b_lo_l = np.maximum(b_map - b_sig, 0.0)   # LOSVD >= 0 everywhere
ax.plot(st.xl, b_map, lw=1.0, ls=':', color='steelblue', alpha=0.8, label='MAP fit')
ax.fill_between(st.xl, b_lo_l, b_map + b_sig,
                alpha=0.35, color='steelblue', label=r'$\pm 1\sigma$ Laplace')

ax.text(0.73, 0.79,
        "Recovered:\n"
        fr"V   = {gh_map['vherm']:+.1f} $\pm$ {laplace['gh_err']['gh_vherm']:.1f}" + " km/s\n"
        fr"$\sigma$   = {gh_map['sherm']:.1f} $\pm$ {laplace['gh_err']['gh_sherm']:.1f}" + " km/s\n"
        fr"$h3$ = {gh_map['h3']:+.4f} $\pm$ {laplace['gh_err']['gh_h3']:.4f}" + "\n"
        fr"$h4$ = {gh_map['h4']:+.4f} $\pm$ {laplace['gh_err']['gh_h4']:.4f}",
        transform=ax.transAxes, fontsize=8, ha='left', va='bottom',
        bbox=dict(boxstyle='round', facecolor='white', alpha=1.0, edgecolor='0.8'))

ax.set_xlabel('velocity (km/s)'); ax.set_ylabel('LOSVD  (km/s)⁻¹')
ax.set_title('Laplace covariance')
ax.legend(fontsize=9, loc='upper left', framealpha=1.0, facecolor='white').set_zorder(20)
ax.set_xlim(cfg.losvd_vmin, cfg.losvd_vmax)

# Panel 2: Bootstrap
ax = axes[1]
ax.fill_between(st.xl, b_lo, b_hi, alpha=0.30, color='tomato',
                label=f'bootstrap 16-84th pct  (n={N_BOOT})')
ax.plot(st.xl, b_median, lw=2.0, color='tomato', zorder=5, label='bootstrap median')


ax.text(0.73, 0.79,
        f"Recovered:\n"
        fr"V   = {gh_med['vherm']:+.1f} $\pm$ {gh_err.get('gh_vherm', float('nan')):.1f}" + " km/s\n"
        fr"$\sigma$   = {gh_med['sherm']:.1f} $\pm$ {gh_err.get('gh_sherm', float('nan')):.1f}" + " km/s\n"
        fr"$h3$ = {gh_med['h3']:+.4f} $\pm$ {gh_err.get('gh_h3', float('nan')):.4f}" + "\n"
        fr"$h4$ = {gh_med['h4']:+.4f} $\pm$ {gh_err.get('gh_h4', float('nan')):.4f}",
        transform=ax.transAxes, fontsize=8, ha='left', va='bottom',
        bbox=dict(boxstyle='round', facecolor='white', alpha=1.0, edgecolor='0.8'))

ax.set_xlabel('velocity (km/s)'); ax.set_ylabel('LOSVD  (km/s)⁻¹')
ax.set_title(f'Bootstrap (n={N_BOOT})')
ax.legend(fontsize=9, loc='upper left', framealpha=1.0, facecolor='white').set_zorder(20)
ax.set_xlim(cfg.losvd_vmin, cfg.losvd_vmax)

plt.tight_layout()
plt.show()

## 7. Built-in diagnostic plots

`kinextract.plotting` ships ready-made diagnostic plots so you don't have to
write custom matplotlib code for routine quick-looks:

- `plot_fit(fit)` / `plot_losvd(fit)` — same idea as the panels built by hand
  in Section 5, in one line each.
- `plot_als_continuum(fit, cfg)` — specifically for `fit_als_continuum=True`
  fits like this one. Shows the data, model, and ALS continuum baseline; a
  continuum-normalized view; and residuals, with detected emission/absorption
  regions shaded and labeled. Passing `cfg` (rather than leaving it out) adds
  labels for the specific lines this fit's masking actually detected, on top
  of the curated reference-line ticks shown either way.

In [ ]:
from kinextract.plotting import plot_fit, plot_losvd, plot_als_continuum

plot_fit(fit)
plot_losvd(fit)
plot_als_continuum(fit, cfg)